# 02. Reading, displaying, dtype, intensity range, and channels

**Author:** Md. Mobarak Karim, Ph.D.  
**Level:** Beginner

## What you will learn
- Distinguish grayscale, RGB, and multichannel arrays
- Understand dtype and intensity range
- Separate display changes from data changes
- Use image-aware conversion and standard file I/O

> **Learning rule:** understand the problem first, then choose the function.

## 1. Inspect before processing

A 2-D array is commonly grayscale. A 3-D array may be RGB **or** scientific multichannel data, so shape alone does not define channel meaning.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import skimage as ski

gray = ski.data.camera()
rgb = ski.data.astronaut()

# Inspect shape, dtype, and range before selecting an algorithm.
for name, image in [("gray", gray), ("rgb", rgb)]:
    print(name, "shape=", image.shape, "dtype=", image.dtype, "range=", (image.min(), image.max()))

## 2. Display settings versus stored data

`vmin` and `vmax` change how values are mapped to the screen. They do **not** rewrite the image array.

In [ ]:
# Display-only contrast: the underlying array stays unchanged.
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].imshow(gray, cmap="gray")
axes[0].set_title("Automatic display")
axes[1].imshow(gray, cmap="gray", vmin=80, vmax=180)
axes[1].set_title("Display range 80–180")
for ax in axes: ax.axis("off")
plt.tight_layout(); plt.show()
print("stored range:", gray.min(), gray.max())

## 3. Convert images intentionally

`astype(float)` changes storage type only. `img_as_float()` follows image dtype conventions and is usually the safer image-processing conversion.

In [ ]:
from skimage.util import img_as_float, img_as_ubyte

# Raw casting preserves 0–255 numerical values but changes dtype.
raw_cast = gray.astype(float)
# Image-aware conversion maps uint8 to the conventional 0–1 float range.
normalized = img_as_float(gray)
print("astype(float):", raw_cast.min(), raw_cast.max())
print("img_as_float :", normalized.min(), normalized.max())
# Convert back only when 8-bit output is actually required.
round_trip = img_as_ubyte(normalized)
print("round-trip identical:", np.array_equal(gray, round_trip))

## 4. Read and write standard files

Use `imageio.v3` for common image files. For complex microscopy formats, use a metadata-aware reader when pixel size, channel identity, z-spacing, or timing matters.

In [ ]:
from pathlib import Path
import imageio.v3 as iio

out = Path("../outputs")
out.mkdir(exist_ok=True)
path = out / "example_crop.png"
# Save a derived image, never the only copy of raw experimental data.
iio.imwrite(path, gray[180:330, 180:330])
loaded = iio.imread(path)
print("loaded:", loaded.shape, loaded.dtype)

## Function-selection guide

| Need | Start with | Why |
|---|---|---|
| Inspect storage | `shape`, `dtype`, `min`, `max` | Avoid wrong assumptions |
| Display grayscale | `plt.imshow(..., cmap="gray")` | Correct 2-D visualization |
| Display-only contrast | `vmin`, `vmax` | Does not rewrite data |
| Normalized float image | `ski.util.img_as_float()` | Image-aware conversion |
| Standard I/O | `imageio.v3.imread/imwrite` | Simple common-file workflow |

## Takeaway

**Choose functions because they solve a specific image problem, and always inspect the result before measuring.**